# AGN corona spectral fitting with current cosipy

This notebook updates the older `AGN_XRay` AGN corona workflow to the current cosipy/3ML fitting pattern used by the Crab spectral fitting tutorial. It expects the full DC4 mock data set to already be binned into a cosipy/histpy HDF5 file, applies separate `GoodTimeInterval` cuts for NGC 4151 and NGC 1068 before fitting, and uses the total DC4 background histogram as one fitted background template per source-specific cut.

The fit includes both requested AGN corona models at once: NGC 4151 and NGC 1068, each modeled as a cutoff-power-law thermal component plus a simple-power-law non-thermal tail.

In [ ]:
from pathlib import Path
import sys

analysis_dir = Path.cwd()
if not (analysis_dir / "agn.yaml").exists():
    repo_notebook_dir = Path("/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN")
    if repo_notebook_dir.exists():
        analysis_dir = repo_notebook_dir

from cosipy import BinnedData
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.util import fetch_wasabi_file
from cosipy.event_selection import GoodTimeInterval

from cosipy.statistics import PoissonLikelihood
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response import (
    BinnedThreeMLModelFolding,
    BinnedInstrumentResponse,
    BinnedThreeMLPointSourceResponse,
)
from cosipy.data_io import EmCDSBinnedData

from histpy import Histogram

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time

import numpy as np
import matplotlib.pyplot as plt

from threeML import PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Cutoff_powerlaw, Powerlaw, Line

%matplotlib inline


## Paths and run options

Set `binned_mock_data_file` to the HDF5 file you create from the full mock unbinned data set. The data and background are both filtered with the final GTI before being projected to `(Em, Phi, PsiChi)` for fitting.

In [ ]:
binned_mock_data_file = Path("/path/to/your/binned_dc4_mock_dataset_3months.hdf5")

total_background_file = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck.hdf5"
)

agn_config = analysis_dir / "agn.yaml"
use_pointing_gti = True
use_earth_occultation_cut = True
max_offaxis = 60 * u.deg

# Add explicit time cuts here as UNIX-second half-open intervals: [start, stop).
# These are intersected with the full background span and, if enabled, the union
# of the per-source pointing GTIs.
manual_time_intervals = [
    # (1836000000.0, 1836100000.0),
]

source_settings = {
    "ngc4151": {
        "label": "NGC 4151",
        "l": 155.07 * u.deg,
        "b": 75.06 * u.deg,
        "thermal_K_at_1keV": 0.15 / (u.cm * u.cm * u.s * u.keV),
        "thermal_index": -1.75,
        "thermal_cutoff": 200 * u.keV,
        "tail_ratio_at_200keV": 0.15,
        "tail_index": -3.8,
    },
    "ngc1068": {
        "label": "NGC 1068",
        "l": 17.2 * u.deg,
        "b": -51.9 * u.deg,
        "thermal_K_at_1keV": 3.1e-1 / (u.cm * u.cm * u.s * u.keV),
        "thermal_index": -1.92,
        "thermal_cutoff": 200 * u.keV,
        "tail_ratio_at_200keV": 0.15,
        "tail_index": -3.8,
    },
}

source_coords = {
    name: SkyCoord(l=settings["l"], b=settings["b"], frame="galactic")
    for name, settings in source_settings.items()
}


## Spacecraft file and response

This follows the same current response-loading pattern as the Crab spectral fitting notebook. If you already have local response/orientation files, replace these paths and skip the downloads.

In [ ]:
orientation_path = analysis_dir / "20280301_3_month_with_orbital_info.ori"
fetch_wasabi_file(
    "COSI-SMEX/DC2/Data/Orientation/20280301_3_month_with_orbital_info.ori",
    output=str(orientation_path),
    checksum="416fcc296fc37a056a069378a2d30cb2",
)
sc_orientation = SpacecraftHistory.open(orientation_path)

response_path = analysis_dir / "ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
fetch_wasabi_file(
    "COSI-SMEX/develop/Data/Responses/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5",
    output=str(response_path),
    checksum="7121f094be50e7bfe9b31e53015b0e85",
)
dr = FullDetectorResponse.open(str(response_path))


## Build Source-Specific GTIs

`GoodTimeInterval` is used for optional explicit time cuts and for source-visibility cuts from the spacecraft history. The notebook keeps separate GTIs for NGC 4151 and NGC 1068. Each source gets its own cut data histogram, cut background histogram, cut spacecraft history, and COSI plugin in the joint fit.

If the two source GTIs overlap, the overlapping binned time intervals will appear in both source-specific datasets. That matches a two-cut workflow, but it means those overlapping data bins contribute to both plugin likelihoods.

In [ ]:
total_background = Histogram.open(total_background_file)
background_time_edges = total_background.axes["Time"].edges.to_value(u.s)
full_span_gti = GoodTimeInterval(
    Time([background_time_edges[0]], format="unix"),
    Time([background_time_edges[-1]], format="unix"),
)

base_gti_parts = [full_span_gti]

if manual_time_intervals:
    manual_gti = GoodTimeInterval(
        Time([start for start, _ in manual_time_intervals], format="unix"),
        Time([stop for _, stop in manual_time_intervals], format="unix"),
    )
    base_gti_parts.append(manual_gti)

source_pointing_gtis = {}
source_analysis_gtis = {}
source_fit_sc_orientations = {}

for name, coord in source_coords.items():
    gti_parts = list(base_gti_parts)

    if use_pointing_gti:
        source_pointing_gtis[name] = GoodTimeInterval.from_pointing_cut(
            coord,
            sc_orientation,
            max_offaxis=max_offaxis,
            earth_occ=use_earth_occultation_cut,
        )
        gti_parts.append(source_pointing_gtis[name])

    source_analysis_gtis[name] = GoodTimeInterval.intersection(*gti_parts)
    source_fit_sc_orientations[name] = sc_orientation.apply_gti(source_analysis_gtis[name])

    gti = source_analysis_gtis[name]
    print(f"{source_settings[name]['label']}: using {len(gti)} GTI interval(s)")
    print(f"  First GTI: {gti.tstart_list[0].unix:.3f} - {gti.tstop_list[0].unix:.3f}")
    print(f"  Last GTI:  {gti.tstart_list[-1].unix:.3f} - {gti.tstop_list[-1].unix:.3f}")


In [ ]:
def contiguous_true_ranges(mask):
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return
    breaks = np.where(np.diff(idx) > 1)[0] + 1
    for group in np.split(idx, breaks):
        yield int(group[0]), int(group[-1]) + 1


def project_histogram_over_gti(hist, gti, labels=("Em", "Phi", "PsiChi")):
    if "Time" not in hist.axes.labels:
        return hist.project(*labels)

    centers = hist.axes["Time"].centers.to_value(u.s)
    selected = np.zeros(len(centers), dtype=bool)
    for start, stop in gti:
        selected |= (centers >= start.unix) & (centers < stop.unix)

    if not np.any(selected):
        raise ValueError("No time bins have centers inside the selected GTI")

    time_axis = hist.axes.labels.index("Time")
    projected = None
    for lo, hi in contiguous_true_ranges(selected):
        indexer = [slice(None)] * hist.ndim
        indexer[time_axis] = slice(lo, hi)
        chunk = hist.slice[tuple(indexer)].project(*labels)
        projected = chunk if projected is None else projected + chunk

    return projected


## Load binned data and total background

`FreeNormBinnedBackground` accepts a dictionary of components. Here each source-specific plugin gets its own `{"total_bkg": ...}` template, sliced with that source's GTI.

In [ ]:
if not binned_mock_data_file.exists():
    raise FileNotFoundError(
        f"Set binned_mock_data_file to your binned full mock data HDF5 file: {binned_mock_data_file}"
    )

agn = BinnedData(agn_config)
agn.load_binned_data_from_hdf5(binned_data=binned_mock_data_file)

data_hists = {}
total_bkgs = {}
bkg_dists = {}

for name, gti in source_analysis_gtis.items():
    data_hists[name] = project_histogram_over_gti(agn.binned_data, gti)
    total_bkgs[name] = project_histogram_over_gti(total_background, gti)
    bkg_dists[name] = {"total_bkg": total_bkgs[name]}

    for label in bkg_dists[name]:
        bkg_dists[name][label] += sys.float_info.min


## COSI 3ML plugin

In [ ]:
def build_cosi_plugin(plugin_name, data_hist, bkg_dist, sc_history):
    data = EmCDSBinnedData(data_hist)
    bkg = FreeNormBinnedBackground(bkg_dist, sc_history=sc_history, copy=False)

    instrument_response = BinnedInstrumentResponse(dr, data)
    psr = BinnedThreeMLPointSourceResponse(
        data=data,
        instrument_response=instrument_response,
        sc_history=sc_history,
        energy_axis=dr.axes["Ei"],
        polarization_axis=dr.axes["Pol"] if "Pol" in dr.axes.labels else None,
        nside=2 * data.axes["PsiChi"].nside,
    )
    response = BinnedThreeMLModelFolding(data=data, point_source_response=psr)
    like_fun = PoissonLikelihood(data, response, bkg)
    plugin = ThreeMLPluginInterface(plugin_name, like_fun, response, bkg)

    for bkg_label in bkg_dist:
        plugin.bkg_parameter[bkg_label] = Parameter(
            bkg_label,
            1.0,
            min_value=0.0,
            max_value=100.0,
            delta=0.05,
            unit=u.Hz,
        )

    return plugin, response, bkg

cosi_plugins = {}
responses = {}
background_models = {}

for name in source_settings:
    plugin_name = f"cosi_{name}"
    cosi_plugins[name], responses[name], background_models[name] = build_cosi_plugin(
        plugin_name,
        data_hists[name],
        bkg_dists[name],
        source_fit_sc_orientations[name],
    )


## Source model

The fit includes both requested AGN corona models at once: NGC 4151 with a 200 keV cutoff-power-law thermal component plus a simple power-law non-thermal tail, and NGC 1068 with the same thermal-plus-tail structure. The tail normalizations are linked to their corresponding thermal normalizations at 200 keV by the ratios from the older notebook.

In [ ]:
def make_cutoff_powerlaw(K, piv, xc, index):
    spectrum = Cutoff_powerlaw()
    spectrum.K.value = K.to_value(1 / (u.cm * u.cm * u.s * u.keV))
    spectrum.piv.value = piv.to_value(u.keV)
    spectrum.xc.value = xc.to_value(u.keV)
    spectrum.index.value = index
    spectrum.K.unit = K.unit
    spectrum.piv.unit = piv.unit
    spectrum.xc.unit = xc.unit
    return spectrum


def make_powerlaw_tail(thermal_spectrum, ratio, tail_index=-3.8, pivot=200 * u.keV):
    tail = Powerlaw()
    K = ratio * thermal_spectrum.evaluate_at(pivot.to_value(u.keV)) / (u.cm * u.cm * u.s * u.keV)
    tail.K.value = K.value
    tail.piv.value = pivot.to_value(u.keV)
    tail.index.value = tail_index
    tail.K.unit = K.unit
    tail.piv.unit = pivot.unit
    return tail


reference_spectra = {}
model_sources = []
component_source_names = []

for name, settings in source_settings.items():
    reference_thermal = make_cutoff_powerlaw(
        K=settings["thermal_K_at_1keV"],
        piv=1 * u.keV,
        xc=settings["thermal_cutoff"],
        index=settings["thermal_index"],
    )
    reference_tail = make_powerlaw_tail(
        reference_thermal,
        ratio=settings["tail_ratio_at_200keV"],
        tail_index=settings["tail_index"],
    )
    reference_spectra[name] = {
        "thermal": reference_thermal,
        "tail": reference_tail,
        "total": reference_thermal + reference_tail,
    }

    thermal_K_at_200 = reference_thermal.evaluate_at(200.0) / (u.cm * u.cm * u.s * u.keV)
    thermal = make_cutoff_powerlaw(
        K=thermal_K_at_200,
        piv=200 * u.keV,
        xc=settings["thermal_cutoff"],
        index=settings["thermal_index"],
    )
    thermal.index.fix = True
    thermal.K.min_value = 1e-10
    thermal.K.max_value = 1e-1
    thermal.xc.min_value = 100
    thermal.xc.max_value = 10000
    thermal.xc.delta = 10

    tail = Powerlaw()
    tail.K.value = max(reference_tail.evaluate_at(200.0), 1e-12)
    tail.piv.value = 200.0
    tail.index.value = settings["tail_index"]
    tail.K.unit = 1 / (u.cm * u.cm * u.s * u.keV)
    tail.piv.unit = u.keV
    tail.K.min_value = 1e-12
    tail.K.max_value = 1e-2
    tail.index.min_value = -5
    tail.index.max_value = 1
    tail.index.delta = 0.25

    thermal_source_name = f"{name}_thermal"
    tail_source_name = f"{name}_tail"
    thermal_source = PointSource(
        thermal_source_name,
        l=settings["l"].value,
        b=settings["b"].value,
        spectral_shape=thermal,
    )
    tail_source = PointSource(
        tail_source_name,
        l=settings["l"].value,
        b=settings["b"].value,
        spectral_shape=tail,
    )
    model_sources.extend([thermal_source, tail_source])
    component_source_names.extend([thermal_source_name, tail_source_name])

model = Model(*model_sources)

for name, settings in source_settings.items():
    link_function = Line(a=0.0, b=settings["tail_ratio_at_200keV"])
    link_function.a.fix = True
    model.link(
        getattr(model, f"{name}_tail").spectrum.main.Powerlaw.K,
        getattr(model, f"{name}_thermal").spectrum.main.Cutoff_powerlaw.K,
        link_function,
    )


## Fit

In [ ]:
plugins = DataList(*cosi_plugins.values())
like = JointLikelihood(model, plugins, verbose=False)
fit_result = like.fit()


## Null likelihood and significance

In [ ]:
def make_null_likelihood():
    null_plugins = {}

    for name in source_settings:
        plugin_name = f"cosi_null_{name}"
        null_plugins[name], _, _ = build_cosi_plugin(
            plugin_name,
            data_hists[name],
            bkg_dists[name],
            source_fit_sc_orientations[name],
        )

    spectrum_null = Powerlaw()
    spectrum_null.K.value = 1e-30
    spectrum_null.index.value = 1.0
    spectrum_null.K.fix = True
    spectrum_null.index.fix = True

    source_null = PointSource(
        "source_null",
        l=source_settings["ngc4151"]["l"].value,
        b=source_settings["ngc4151"]["b"].value,
        spectral_shape=spectrum_null,
    )
    model_null = Model(source_null)
    plugins_null = DataList(*null_plugins.values())
    like_null = JointLikelihood(model_null, plugins_null, verbose=False)
    like_null.fit()
    return like_null


def sum_statistic(results, plugin_names):
    stat_frame = results.get_statistic_frame()["-log(likelihood)"]
    return sum(stat_frame[name] for name in plugin_names)

like_null = make_null_likelihood()
fit_stat = sum_statistic(like.results, [f"cosi_{name}" for name in source_settings])
null_stat = sum_statistic(like_null.results, [f"cosi_null_{name}" for name in source_settings])
TS = 2 * (null_stat - fit_stat)
print("TS:", TS)
print("Significance:", np.sqrt(TS))


## Error propagation and plots

In [ ]:
results = like.results
print(results.display())

component_errors = {}
for source_name in component_source_names:
    parameters = {
        par.name: results.get_variates(par.path)
        for par in results.optimized_model[source_name].parameters.values()
        if par.free
    }
    component_errors[source_name] = results.propagate(
        results.optimized_model[source_name].spectrum.main.shape.evaluate_at,
        **parameters,
    )


In [ ]:
energy = np.geomspace(200 * u.keV, 5 * u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = np.zeros_like(energy)
component_flux_median = {source_name: np.zeros_like(energy) for source_name in component_source_names}
source_flux_inj = {name: np.zeros_like(energy) for name in source_settings}

for i, e in enumerate(energy):
    combined = None
    for source_name in component_source_names:
        flux_component = component_errors[source_name](e)
        component_flux_median[source_name][i] = flux_component.median
        combined = flux_component if combined is None else combined + flux_component

    flux_median[i] = combined.median
    flux_lo[i], flux_hi[i] = combined.equal_tail_interval(cl=0.68)

    injected_total = 0.0
    for name in source_settings:
        source_flux = reference_spectra[name]["total"].evaluate_at(e)
        source_flux_inj[name][i] = source_flux
        injected_total += source_flux
    flux_inj[i] = injected_total

binned_energy_edges = next(iter(data_hists.values())).axes["Em"].edges.value
binned_energy = 0.5 * (binned_energy_edges[1:] + binned_energy_edges[:-1])

expectations = {name: response.expectation() for name, response in responses.items()}
expectation_backgrounds = {name: bkg.expectation(copy=True) for name, bkg in background_models.items()}


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

sigma = np.sqrt(TS)
ax.plot(energy, energy * energy * flux_inj, color="black", ls=":", lw=2, label="Reference total model")
ax.plot(energy, energy * energy * flux_median, color="tab:red", label=f"Best fit total ({sigma:.2f} sigma)")
ax.fill_between(energy, energy * energy * flux_lo, energy * energy * flux_hi, color="tab:red", alpha=0.25, label="68% interval")

colors = {"ngc4151": "tab:orange", "ngc1068": "tab:blue"}
for name, settings in source_settings.items():
    thermal_name = f"{name}_thermal"
    tail_name = f"{name}_tail"
    ax.plot(
        energy,
        energy * energy * source_flux_inj[name],
        color=colors[name],
        ls=":",
        lw=1.5,
        label=f"{settings['label']} reference",
    )
    ax.plot(
        energy,
        energy * energy * component_flux_median[thermal_name],
        color=colors[name],
        ls="--",
        label=f"{settings['label']} thermal fit",
    )
    ax.plot(
        energy,
        energy * energy * component_flux_median[tail_name],
        color=colors[name],
        ls="-.",
        label=f"{settings['label']} non-thermal fit",
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energy (keV)")
ax.set_ylabel(r"Energy flux (keV cm$^{-2}$ s$^{-1}$)")
ax.set_title("NGC 4151 + NGC 1068")
ax.legend(frameon=False, fontsize=9)
plt.show()


In [ ]:
fig, axes = plt.subplots(
    len(source_settings),
    1,
    figsize=(9, 5 * len(source_settings)),
    constrained_layout=True,
    sharex=True,
)

if len(source_settings) == 1:
    axes = [axes]

for ax, (name, settings) in zip(axes, source_settings.items()):
    expectation = expectations[name]
    expectation_bkg = expectation_backgrounds[name]

    source_counts = expectation.project("Em").to_dense(copy=False).contents
    background_counts = expectation_bkg.project("Em").to_dense(copy=False).contents
    total_expectation_counts = source_counts + background_counts
    data_counts = data_hists[name].project("Em").to_dense(copy=False).contents

    ax.stairs(total_expectation_counts, binned_energy_edges, color="tab:purple", label="Best fit source + total background")
    ax.errorbar(
        binned_energy,
        total_expectation_counts,
        yerr=np.sqrt(total_expectation_counts),
        color="tab:purple",
        linewidth=0,
        elinewidth=1,
    )
    ax.stairs(data_counts, binned_energy_edges, color="black", ls=":", label="GTI-cut mock data")
    ax.errorbar(
        binned_energy,
        data_counts,
        yerr=np.sqrt(data_counts),
        color="black",
        linewidth=0,
        elinewidth=1,
    )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylabel("Counts")
    ax.set_title(f"{settings['label']} source-specific cut")
    ax.legend(frameon=False)

axes[-1].set_xlabel("Energy (keV)")
plt.show()


## Optional profile scans

In [ ]:
ngc4151_thermal_k_path = "ngc4151_thermal.spectrum.main.Cutoff_powerlaw.K"
ngc4151_thermal_xc_path = "ngc4151_thermal.spectrum.main.Cutoff_powerlaw.xc"
ngc4151_tail_idx_path = "ngc4151_tail.spectrum.main.Powerlaw.index"
ngc1068_thermal_k_path = "ngc1068_thermal.spectrum.main.Cutoff_powerlaw.K"
ngc1068_thermal_xc_path = "ngc1068_thermal.spectrum.main.Cutoff_powerlaw.xc"
ngc1068_tail_idx_path = "ngc1068_tail.spectrum.main.Powerlaw.index"


def get_profile(like, path, scale_low=None, scale_high=None, delta_low=None, delta_high=None, log=False, n_steps=40):
    best = like.likelihood_model[path].value
    pmin, pmax = like.likelihood_model[path].bounds
    pmin = -np.inf if pmin is None else pmin
    pmax = np.inf if pmax is None else pmax

    if scale_low is not None:
        low = max(best * scale_low, pmin)
    else:
        low = max(best + delta_low, pmin)

    if scale_high is not None:
        high = min(best * scale_high, pmax)
    else:
        high = min(best + delta_high, pmax)

    x, _, prof, fig = like.get_contours(
        path,
        param_1_minimum=low,
        param_1_maximum=high,
        param_1_n_steps=n_steps,
        progress=True,
        log=(log,),
    )
    delta_ts = 2 * (prof - prof.min())
    return x, delta_ts, fig

# Examples:
# x_k, dts_k, _ = get_profile(like, ngc4151_thermal_k_path, scale_low=0.3, scale_high=3.0, log=True)
# x_xc, dts_xc, _ = get_profile(like, ngc1068_thermal_xc_path, scale_low=0.3, scale_high=3.0, log=True)
# x_tail_idx, dts_tail_idx, _ = get_profile(like, ngc4151_tail_idx_path, delta_low=-1.0, delta_high=1.0)
